In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

In [2]:
# 하이퍼파라미터 설정
BATCH_SIZE=32
EPOCHS=10
LR =0.001
NUM_CLASSES =2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# GPU 사용 가능 여부 확인
print(f"GPU 사용 가능 여부: {torch.cuda.is_available()}")

# 사용 가능한 GPU 개수 및 이름 확인
if torch.cuda.is_available():
    print(f"GPU 개수: {torch.cuda.device_count()}")
    print(f"현재 GPU 이름: {torch.cuda.get_device_name(0)}")


GPU 사용 가능 여부: True
GPU 개수: 4
현재 GPU 이름: NVIDIA TITAN Xp


In [4]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 사용할 GPU 번호 입력
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import glob 
import os 

# 경로 지정 
root='./'
image_folder_path = os.path.join(root, 'product_images') 

# images 폴더가 없으면 생성(이미 존재하면 건너뜀)
if not os.path.exists(image_folder_path):
    os.makedirs(image_folder_path)

import kaggle
kaggle.api.dataset_download_files('ravirajsinh45/real-life-industrial-dataset-of-casting-product', path=image_folder_path, unzip=True)

print(image_folder_path)
print(glob.glob(image_folder_path+'/*')) 

Dataset URL: https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product
./product_images
['./product_images/casting_data', './product_images/casting_512x512']


In [7]:
TRAIN_DIR = os.path.join(image_folder_path, "casting_data", "casting_data","train")
TEST_DIR = os.path.join(image_folder_path, "casting_data", "casting_data", "test")

print("Train 경로:", TRAIN_DIR)
print("TEST 경로:", TEST_DIR)

Train 경로: ./product_images/casting_data/casting_data/train
TEST 경로: ./product_images/casting_data/casting_data/test


In [10]:
# VGGNet 19-layer (VGG-19) 정의 
class VGG19(nn.Module):
    def __init__(self, num_classes=1000):
        super(VGG19, self).__init__()

        # Convolution + ReLU layers (3x3 convolution, padding=1)
        # nn.Sequential는 "레이어를 순서대로 담는 상자"이다.

        self.features = nn.Sequential(
            # Block 1 : 2 conv layers + maxpool
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 2 conv layers + maxpool
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 4 conv layers + maxpool
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4 : 4 conv layers + maxpool
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 5 : 4 conv layers + maxpool
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Fully Connected layers
        self.classifier = nn.Sequential(
            nn.Linear(512*7*7, 4096), # 입력 크기 : 마지막 feature map 7x7
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, num_classes)
        )

    def forward(self,x):
        x=self.features(x) # Convolution & Pooling
        x=x.view(x.size(0),-1) # Flatten
        x=self.classifier(x) # Fully Connected Network
        return x
    
model = VGG19(num_classes=2)
# print(model)